# COMP9085 Week 10 Laboratory
## Random forests, honest importances, and a pretrained protein embedding

**Two hours.** In the lecture you saw why averaging many unstable trees works, how to read out-of-bag error, why the two
standard feature importances can mislead in opposite ways, and what happened when a forest was given ESM-2 protein
embeddings instead of features a biologist designed. Today you reproduce each of those results yourself, and then write
the recommendation a collaborator would actually read.

**Data.**
* **WDBC** (UCI Breast Cancer Wisconsin, Diagnostic): 569 tumours, 30 measurements of cell nuclei from a fine-needle
  aspirate image (Weeks 2, 3, 7 and 9). It ships with scikit-learn, so no download is needed. Label: malignant (1) or
  benign (0).
* **The Week 7 proteins**: 2,000 human Swiss-Prot proteins (UniProt release 2026_03, one per UniRef50 cluster), two
  labels (signal peptide; transmembrane), the 23 classical descriptors and the ESM-2 (8M) embeddings. Files in `data/`
  (provenance in `data/README.md`). The train/test split is **exactly the Week 7 split**.

**Learning objectives.** By the end you should be able to:
1. derive and check by simulation the share of samples in a bootstrap sample;
2. compare a single tree, bagged trees, a random forest and logistic regression under one protocol;
3. read an out-of-bag error curve and choose the number of trees;
4. explain, with your own measurements, why deep single trees overfit and forests of deep trees do not;
5. measure impurity and permutation importance and expose their correlated-feature and noise failure modes;
6. compare engineered features with a frozen protein-language-model embedding, with an uncertainty interval;
7. recommend a model on more than accuracy.

**How to work.** Core tasks 1–9 fit in two hours (times are a guide). Extensions A–C are optional. Code cells marked
`# your code here` are yours; cells marked `# provided` are complete. Every task ends with a question to answer in the
markdown cell below it: **the written answers matter more than the code.** The matching website exercises are on the
Week 10 page: https://hafli.github.io/comp9085-ml-biology/weeks/week10.html

**Deliverable.** This notebook, run top to bottom (**Kernel → Restart & Run All**), with every
question answered in the markdown cells.

**Runs in** Google Colab or any Python 3.10+ environment with numpy, pandas, matplotlib and scikit-learn (≥ 1.3).
No GPU and no deep-learning package are needed: the embeddings are precomputed. The complete notebook runs in about
two to four minutes on a laptop; Task 8 takes most of that.

In [ ]:
# provided: setup
import time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
import sklearn
warnings.filterwarnings("ignore")
SEED = 42
print("scikit-learn", sklearn.__version__)

# WDBC: 1 = malignant (scikit-learn codes benign as 1, so we flip it)
wdbc = load_breast_cancer()
X = wdbc.data
names = [n.replace(" ", "_") for n in wdbc.feature_names]
y = (wdbc.target == 0).astype(int)
n, p = X.shape
print(X.shape, "tumours x features;", y.sum(), "malignant,", n - y.sum(), "benign")

def cv5():
    return StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def logreg():
    return Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=5000, random_state=SEED))])

def knn5():
    return Pipeline([("sc", StandardScaler()), ("m", KNeighborsClassifier(n_neighbors=5))])

def rf(n_trees=500, **kw):
    return RandomForestClassifier(n_estimators=n_trees, random_state=SEED, n_jobs=-1, **kw)

# The Week 7 proteins: local data/ folder, or the course website (for Colab)
URL = "https://hafli.github.io/comp9085-ml-biology/labs/week10/data/"
FILES = ["proteins_classical_features.csv", "esm2_t6_8M_embeddings.csv.gz", "esm2_t6_8M_embeddings_shuffled.csv.gz"]
DATA = next((q for q in [Path("data"), Path("../data")] if (q / FILES[0]).exists()), None)
if DATA is None:
    import urllib.request
    DATA = Path("data"); DATA.mkdir(exist_ok=True)
    for f in FILES:
        urllib.request.urlretrieve(URL + f, DATA / f)
prot = pd.read_csv(DATA / "proteins_classical_features.csv")
emb = pd.read_csv(DATA / "esm2_t6_8M_embeddings.csv.gz").set_index("accession").loc[prot.accession].values
CLASSICAL = [c for c in prot.columns if c.startswith("frac_")] + ["log_length", "max_hydropathy_19", "nterm_hydropathy_25"]
Xc = prot[CLASSICAL].values
sp, tm = prot.has_signal_peptide.values, prot.is_transmembrane.values
tr, te = train_test_split(np.arange(len(prot)), test_size=0.25, stratify=2 * tm + sp, random_state=SEED)   # the Week 7 split
print(len(prot), "proteins;", Xc.shape[1], "classical features;", emb.shape[1], "embedding dimensions;",
      len(tr), "train /", len(te), "test")

## Task 1. The bootstrap, by derivation and by simulation (10 min)

**1a.** Draw one bootstrap sample of the 569 row indices (`rng.integers(0, n, n)`). How many distinct tumours does it
contain, and how many are out of bag?

**1b.** Repeat 2,000 times with `rng = np.random.default_rng(SEED)` and report the mean and SD of the share of distinct
tumours. Compare with the exact value 1 − (1 − 1/n)ⁿ and with the limit 1 − 1/e.

In [ ]:
rng = np.random.default_rng(SEED)
# your code here


**Question 1.** Why does every tumour end up with an out-of-bag prediction once a forest has a few dozen trees? Roughly how many of 500 trees vote on each tumour's OOB prediction?

*Your answer here.*

## Task 2. The model ladder (15 min)

Using `cv5()` (5-fold stratified CV, seed 42), report the mean and SD of accuracy and the mean ROC AUC for:
1. `DecisionTreeClassifier(random_state=SEED)` (unpruned);
2. `BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, random_state=SEED, n_jobs=-1)`;
3. `rf()` (500 trees, 5 features per split);
4. `logreg()`.

Write a function `cv_eval(model, X, y)` that fits a fresh clone on each training fold and scores the held-out fold
using `predict_proba`. Collect the results in a DataFrame `ladder`.

In [ ]:
def cv_eval(model, X, y):
    # your code here
    pass


**Question 2.** Explain the ordering. Why does the AUC jump more than the accuracy from one tree to bagging? What would you report if a reviewer asked why you used a forest at all?

*Your answer here.*

## Task 3. Out-of-bag error and the number of trees (15 min)

**3a.** For `max_features` m in 1, 5 and 30, grow one forest with `warm_start=True, oob_score=True` and increase
`n_estimators` through `grid` (below), refitting each time. Record `1 - oob_score_` from 15 trees upwards (below that,
some tumours have no out-of-bag vote). Plot the three curves against the number of trees (log scale).

**3b.** Fit `rf(500, oob_score=True)` on all 569 tumours and compare `oob_score_` with the 5-fold CV accuracy of the
forest from Task 2.

In [ ]:
grid = [1, 2, 3, 5, 8, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200, 300, 400, 500]
# your code here


**Question 3.** Does the OOB error ever turn upwards as trees are added? After how many trees would you stop, and why is `n_estimators` not something you tune by cross-validation?

*Your answer here.*

## Task 4. Depth: one tree against a forest (10 min)

For `max_depth` in 1, 2, 3, 4, 6, 8, 12 and `None`, compute the mean **training** and **cross-validated** accuracy over
the five folds of `cv5()`, for `DecisionTreeClassifier(max_depth=d, random_state=SEED)` and for `rf(300, max_depth=d)`.
Plot all four curves.

In [ ]:
depths = [1, 2, 3, 4, 6, 8, 12, None]
# your code here


**Question 4.** At which depth is the single tree best, and what happens beyond it? Why does the forest of fully grown trees not suffer in the same way, even though its training accuracy is also 100%?

*Your answer here.*

## Task 5. Robustness to irrelevant features and to scaling (10 min)

**5a.** Generate 100 columns of pure noise with `NZ = np.random.default_rng(SEED).normal(size=(n, 300))[:, :100]`
(the first 100 of 300, as in the lecture) and append them to `X`. Compare the CV accuracy of `rf(300)`, `knn5()` and
`logreg()` with and without the noise.

**5b.** Compare `rf()` and `KNeighborsClassifier(5)` on raw `X` with their standardised versions.

In [ ]:
# your code here


**Question 5.** Which model loses most from noise, and why? Why does standardising change nothing for the forest? Name a biological setting where the forest's robustness to irrelevant features would stop being enough.

*Your answer here.*

## Task 6. Two importances, correlated features and noise (25 min)

Use a 70/30 stratified split: `Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)`,
and fit `f = rf(500).fit(Xtr, ytr)`.

**6a.** Tabulate MDI (`f.feature_importances_`) and permutation importance on the **test** data
(`permutation_importance(f, Xte, yte, n_repeats=30, random_state=SEED, scoring="roc_auc", n_jobs=-1)`). Show the top
10 of each.

**6b.** Find the most correlated pair of features. Then define the *size group*: every feature whose name contains
radius, perimeter or area (nine features). Report their mean absolute correlation.

**6c.** Permute the nine size features **together** (same row permutation for all nine; 30 repeats with
`np.random.default_rng(SEED)`) and report the mean drop in test AUC. Compare it with the sum of their individual
permutation importances.

**6d.** Keep only `worst_area` from the size group, refit `rf(500)` and report its MDI and rank before and after.

**6e.** Append one column of continuous noise and one of binary noise
(`g = np.random.default_rng(SEED); g.normal(size=n)`, then `g.integers(0, 2, n)`), redo the split and the forest, and
report both importances for the two noise columns.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
f = rf(500).fit(Xtr, ytr)
# your code here


**Question 6.** Your collaborator wants to write "worst perimeter is the most important feature for malignancy". Using your numbers from 6a–6e, write the two or three sentences you would put in the paper instead.

*Your answer here.*

## Task 7. Is the ranking stable? (10 min)

Refit `RandomForestClassifier(500, random_state=s, n_jobs=-1)` on `Xtr, ytr` for seeds s = 0 to 9. For each seed, record
the top-5 features by MDI. How many different features ever appear in the top 5, and how often is each one there?
Is the *order* of the top three the same every time?

In [ ]:
# your code here


**Question 7.** What does this tell you about reporting a single ranking from a single seed? How would you report the top features instead?

*Your answer here.*

## Task 8. Engineered features or a frozen ESM-2 embedding? (25 min)

The Week 7 proteins, the Week 7 split (`tr`, `te`), the Week 7 protocol: choose settings by 5-fold CV ROC AUC on the
1,500 training proteins, then use the 500 test proteins **once** per pipeline.

For each task (`sp` = signal peptide, `tm` = transmembrane) and each representation (`Xc`, 23 classical descriptors;
`emb`, 320 ESM-2 dimensions), fit:
* a random forest with `GridSearchCV(RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1), grid, scoring="roc_auc", cv=cv5())`,
  where to save time `grid` is `{"max_features": ["sqrt", 0.1, 0.3], "min_samples_leaf": [1]}` for the classical
  descriptors and `{"max_features": ["sqrt"], "min_samples_leaf": [1, 5]}` for ESM-2;
* logistic regression: `GridSearchCV(logreg(), {"m__C": [0.01, 0.1, 1.0]}, scoring="roc_auc", cv=cv5(), n_jobs=-1)`.

Record the chosen settings, the CV AUC, and the test ROC AUC and average precision. Then compute a **paired bootstrap**
95% interval for *test AUC(ESM-2 forest) − test AUC(classical forest)*: 2,000 resamples of the 500 test proteins with
`np.random.default_rng(SEED)`, skipping any resample that contains only one class. Finally, show the five most
important classical features (permutation importance on the test proteins, 10 repeats) for each task.

*(The lecture used a slightly larger grid; every setting it chose is inside these grids, so your test numbers should match the lecture's.)*

In [ ]:
REPS = {"classical": Xc, "esm2": emb}
GRIDS = {"classical": {"max_features": ["sqrt", 0.1, 0.3], "min_samples_leaf": [1]},
         "esm2": {"max_features": ["sqrt"], "min_samples_leaf": [1, 5]}}
# your code here


**Question 8.** State what you can and cannot claim from these results. Why might the forest be the weakest head on the ESM-2 embedding while logistic regression does well? Do the classical forest's top features make biological sense, and are they a discovery?

*Your answer here.*

## Task 9. The recommendation (10 min)

A clinical-genomics group asks which pipeline to use to flag likely transmembrane proteins in a new set of 3,000 human
proteins they will later test in the lab. They have no GPU and one bioinformatician. Fill in the table (your numbers,
plus your judgement for the other columns) and write a recommendation of at most 120 words.

| Pipeline | Test AUC (TM) | Labels needed | Compute | Interpretability | Validation burden |
|---|---|---|---|---|---|
| Classical + forest | | | | | |
| Classical + logistic | | | | | |
| ESM-2 + logistic | | | | | |
| ESM-2 + forest | | | | | |
| Fine-tuned ESM-2 (not run) | | | | | |

**Question 9.** Your recommendation (at most 120 words).

*Your answer here.*

## Optional extensions

**A. Test the axis-aligned hypothesis.** Standardise the classical descriptors on the training proteins, rotate them
with a random orthogonal matrix (`Q, _ = np.linalg.qr(np.random.default_rng(0).normal(size=(23, 23)))`), and compare
the transmembrane test AUC of `rf(500)` and `logreg()` before and after rotation. A rotation keeps every distance and
every linear relationship, but mixes the coordinates.

**B. Optimism of the best CV score.** On WDBC, use an outer `cv5()` and inside each training fold run
`GridSearchCV(rf(300), {"max_features": [1, 2, 5, 10, 20, 30], "min_samples_leaf": [1, 5, 10]}, cv=StratifiedKFold(5, shuffle=True, random_state=0))`.
Compare the mean of the inner best scores with the outer (nested) accuracy.

**C. The shuffled-sequence control.** Load `esm2_t6_8M_embeddings_shuffled.csv.gz` (ESM-2 of each sequence with its
residues shuffled). Fit a forest on it for the signal-peptide task with the lecture's full grid
(`{"max_features": ["sqrt", 0.1, 0.3], "min_samples_leaf": [1, 5]}`) and compare its test AUC with the real embedding.
What does the gap tell you?

In [ ]:
# your code here
